In [1]:
import scanpy as sc
import pandas as pd 

In [54]:
# annotation
anno = pd.read_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Snapatac2/Oligo/atac-meta.csv", index_col=0)
anno

,sample2,barcode2,celltype
AAACAGCCATCGTTCT-1-0,GW6,AAACAGCCATCGTTCT-1,NPC
AAACCGAAGGTCCACA-1-0,GW6,AAACCGAAGGTCCACA-1,NPC
AAAGGACGTGGACCTG-1-0,GW6,AAAGGACGTGGACCTG-1,NPC
AACCGCTCATGTCAGC-1-0,GW6,AACCGCTCATGTCAGC-1,NPC
AACCTCCTCAAGGACA-1-0,GW6,AACCTCCTCAAGGACA-1,NPC
...,...,...,...
TTTGTGGCAAGTCGCT-1-7,GW20,TTTGTGGCAAGTCGCT-1,dIPC
TTTGTGGCACGTAATT-1-7,GW20,TTTGTGGCACGTAATT-1,OPC
TTTGTGTTCCGCATGA-1-7,GW20,TTTGTGTTCCGCATGA-1,dIPC
TTTGTTGGTCACAAAT-1-7,GW20,TTTGTTGGTCACAAAT-1,OPC


In [55]:
adata_raw = sc.read_h5ad('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P2/data/scMulti-omics/1.hscAdata_raw.h5ad')
adata_ref = adata_raw[adata_raw.obs.index.isin(anno.index)]
adata_ref.obs = adata_ref.obs.join(anno)
adata_ref.obs = adata_ref.obs[['sample2', 'barcode2', 'celltype']]
# target_celltypes = [
#     "NPC_proliferative", "NPC", "dIPC", "vIPC", "dorsal_glia", "ventral_glia", "ExN_naive", "ExN_immature_1", "ExN_immature_2", "ExN_immature_3", "ExN_mature_1", "ExN_mature_2", "ExN_mature_3",
#     "InN_naive", "InN_immature_1", "InN_immature_2", "InN_mature_1", "InN_mature_2", "InN_ventral_like", "ventral_unknow", "v0_1", "v2a_b", "MN1", "MN2", "OPC", "Ependymal", "FP", "RP", "Microglia", "VLMC"#, "EC"
# ]
# # 筛选adata对象中指定细胞类型的细胞
# adata_ref = adata_ref[adata_ref.obs['celltype'].isin(target_celltypes)].copy()
adata_ref

AnnData object with n_obs × n_vars = 9282 × 36601
    obs: 'sample2', 'barcode2', 'celltype'
    var: 'gene_ids', 'feature_types'

In [15]:
adata_ref.raw = adata_ref
# sc.pp.normalize_total(adata_ref, target_sum=1e4)
# sc.pp.log1p(adata_ref)
adata_ref.obs = adata_ref.obs.rename(lambda x: f"{x}___cisTopic", axis=0)


In [16]:
adata_ref.obs

,sample2,barcode2,celltype
AAACAGCCATCGTTCT-1-0___cisTopic,GW6,AAACAGCCATCGTTCT-1,NPC
AAACCGAAGGTCCACA-1-0___cisTopic,GW6,AAACCGAAGGTCCACA-1,NPC
AAAGGACGTGGACCTG-1-0___cisTopic,GW6,AAAGGACGTGGACCTG-1,NPC
AACCGCTCATGTCAGC-1-0___cisTopic,GW6,AACCGCTCATGTCAGC-1,NPC
AACCTCCTCAAGGACA-1-0___cisTopic,GW6,AACCTCCTCAAGGACA-1,NPC
...,...,...,...
TTTGTGGCAAGTCGCT-1-7___cisTopic,GW20,TTTGTGGCAAGTCGCT-1,dIPC
TTTGTGGCACGTAATT-1-7___cisTopic,GW20,TTTGTGGCACGTAATT-1,OPC
TTTGTGTTCCGCATGA-1-7___cisTopic,GW20,TTTGTGTTCCGCATGA-1,dIPC
TTTGTTGGTCACAAAT-1-7___cisTopic,GW20,TTTGTTGGTCACAAAT-1,OPC


In [17]:
adata_ref.write('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Oligo/data/oligo.h5ad')

In [18]:
adata_ref.raw.to_adata().obs

,sample2,barcode2,celltype
AAACAGCCATCGTTCT-1-0___cisTopic,GW6,AAACAGCCATCGTTCT-1,NPC
AAACCGAAGGTCCACA-1-0___cisTopic,GW6,AAACCGAAGGTCCACA-1,NPC
AAAGGACGTGGACCTG-1-0___cisTopic,GW6,AAAGGACGTGGACCTG-1,NPC
AACCGCTCATGTCAGC-1-0___cisTopic,GW6,AACCGCTCATGTCAGC-1,NPC
AACCTCCTCAAGGACA-1-0___cisTopic,GW6,AACCTCCTCAAGGACA-1,NPC
...,...,...,...
TTTGTGGCAAGTCGCT-1-7___cisTopic,GW20,TTTGTGGCAAGTCGCT-1,dIPC
TTTGTGGCACGTAATT-1-7___cisTopic,GW20,TTTGTGGCACGTAATT-1,OPC
TTTGTGTTCCGCATGA-1-7___cisTopic,GW20,TTTGTGTTCCGCATGA-1,dIPC
TTTGTTGGTCACAAAT-1-7___cisTopic,GW20,TTTGTTGGTCACAAAT-1,OPC


In [2]:
adata_ref = sc.read_h5ad('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Oligo/data/oligo.h5ad')

In [4]:
adata = adata_ref

In [6]:
# Saving count data
adata.layers["counts"] = adata.X.copy()
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=10000, batch_key="sample2")
sc.tl.pca(adata)
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=30)
sc.external.pp.bbknn(adata, 
                     batch_key="sample2",
                     n_pcs = 50,
                     use_annoy= False,
                     pynndescent_n_neighbors = 40
                    )  # running bbknn 1.3.6

sc.tl.umap(adata, 
           min_dist = 0.5, 
           spread = 0.8)

/cluster2/huanglab/jiamao/conda/envs/scenicplus/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:475: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/cluster2/huanglab/jiamao/conda/envs/scenicplus/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:475: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/cluster2/huanglab/jiamao/conda/envs/scenicplus/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:475: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/cluster2/huanglab/jiamao/conda/envs/scenicplus/lib/python3.1

In [11]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata, groupby="celltype", method="wilcoxon",key_added="rank_genes_celltype")

In [12]:
adata

AnnData object with n_obs × n_vars = 9282 × 36601
    obs: 'sample2', 'barcode2', 'celltype'
    var: 'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'umap', 'rank_genes_groups', 'rank_genes_celltype'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [13]:
marker_cell_type = sc.get.rank_genes_groups_df(adata, group=None, key = 'rank_genes_celltype', log2fc_min =0.25, pval_cutoff=0.01)
marker_cell_type

,group,names,scores,logfoldchanges,pvals,pvals_adj
0,COP,EPB41L2,60.996906,19.335531,0.000000,0.000000
1,COP,CADM2,59.677925,65.867020,0.000000,0.000000
2,COP,BCAS1,55.088047,20.782721,0.000000,0.000000
3,COP,GALNT13,54.631748,15.286365,0.000000,0.000000
4,COP,DSCAM,54.072853,20.422560,0.000000,0.000000
...,...,...,...,...,...,...
9470,dIPC,RPL18,3.479478,0.767439,0.000502,0.008836
9471,dIPC,AC079304.1,3.456160,4.644388,0.000548,0.009573
9472,dIPC,CLIP2,3.451405,0.733770,0.000558,0.009734
9473,dIPC,TSPAN3,3.448085,0.507971,0.000565,0.009841


In [ ]:
marker_cell_type.to_csv('oligo_marker_celltype_de.csv', index=False)